# Train model YOLO cho hệ thống vi phạm đèn đỏ (Google Colab GPU)

Notebook này train lại các model cho **mạnh hơn** trên GPU miễn phí của Colab
(máy laptop i5-4210U không có GPU nên không train tại chỗ được).

**4 model của dự án:**
| Model | Nhiệm vụ | Class |
|-------|----------|-------|
| `vehicle.pt` | nhận diện xe | cars, motorbike |
| `traffic_light.pt` | nhận diện đèn | red, green, yellow |
| `license_plate_detection.pt` | tìm vùng biển số | license_plate |
| `license_plate_ocr.pt` | đọc ký tự biển số | 0-9, A-Z |

**Cách dùng:**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Chạy lần lượt từng cell.
3. Sửa `DATASET` (link Roboflow) cho model bạn muốn train.
4. Tải file `best.pt` về, đổi tên và bỏ vào thư mục `models/...` trên máy.

> ⚠️ Colab free ngắt session sau vài giờ. Lưu checkpoint về Google Drive (cell cuối).


## 1. Kiểm tra GPU + cài thư viện

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow
import torch, ultralytics
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
ultralytics.checks()

## 2. Lấy dataset

Có 3 cách. Chọn 1 cách phù hợp với model đang train.

### Cách A — Roboflow (khuyến nghị, có sẵn nhiều dataset VN)
Vào https://roboflow.com → tìm dataset (ví dụ từ khoá: *vietnam license plate*, *traffic light*,
*vehicle detection*) → Download → chọn định dạng **YOLOv8** → copy đoạn code `roboflow` nó cho.

Gợi ý từ khoá tìm dataset trên Roboflow Universe / Kaggle:
- Xe: "vehicle detection", "car motorbike vietnam"
- Đèn: "traffic light detection"
- Biển số VN: "vietnam license plate", "license plate recognition"
- Ký tự biển số: "license plate ocr characters", "anpr characters"


In [ ]:
# --- CÁCH A: Roboflow ---
# Dán đoạn code Roboflow cấp cho bạn vào đây. Ví dụ mẫu (ĐỔI key/workspace/project):
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(1).download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)

In [ ]:
# --- CÁCH B: Kaggle ---
# Cần upload kaggle.json (Account -> Create API Token) rồi:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <owner>/<dataset-name> && unzip -q *.zip -d data
# DATA_YAML = "data/data.yaml"

In [ ]:
# --- CÁCH C: Tự upload zip dataset (đã theo cấu trúc YOLO) ---
# from google.colab import files
# up = files.upload()   # chọn dataset.zip
# !unzip -q dataset.zip -d data
# DATA_YAML = "data/data.yaml"

## 3. Train

Chọn model nền theo nhu cầu:
- `yolov8n.pt` → **nhẹ nhất**, chạy nhanh trên CPU laptop (khuyến nghị cho máy bạn).
- `yolov8s.pt` → cân bằng.
- `yolov8m.pt` → chính xác cao nhưng chậm trên CPU.

Tăng `epochs` để model khoẻ hơn (100-200). `imgsz=640` là chuẩn.


In [ ]:
from ultralytics import YOLO

BASE   = "yolov8n.pt"    # nhẹ cho CPU laptop; đổi yolov8s/yolov8m nếu cần chính xác hơn
EPOCHS = 150
IMGSZ  = 640
NAME   = "vehicle_v2"    # đổi tên theo model đang train

model = YOLO(BASE)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=30,          # early stop nếu 30 epoch không cải thiện
    device=0,             # GPU
    augment=True,         # bật augmentation -> model tổng quát hơn
    mosaic=1.0, mixup=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.5, fliplr=0.5,
    name=NAME,
)

## 4. Đánh giá + xem kết quả

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50, "| mAP50-95:", metrics.box.map)

# Xem đường cong + ảnh dự đoán mẫu
from IPython.display import Image, display
import glob, os
run_dir = f"runs/detect/{NAME}"
for p in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    fp = os.path.join(run_dir, p)
    if os.path.exists(fp):
        display(Image(fp, width=640))

## 5. Tải model về máy

File cần lấy: `runs/detect/<NAME>/weights/best.pt`

Sau khi tải về, đổi tên và bỏ vào đúng thư mục trên laptop:
| Train model gì | Đổi tên thành | Bỏ vào |
|---|---|---|
| xe | `vehicle.pt` | `models/vehicle/` |
| đèn | `traffic_light.pt` | `models/traffic_light/` |
| vùng biển số | `license_plate_detection.pt` | `models/license_plate/` |
| ký tự biển số | `license_plate_ocr.pt` | `models/license_plate/` |


In [ ]:
from google.colab import files
best = f"runs/detect/{NAME}/weights/best.pt"
print("Tải:", best)
files.download(best)

### (Tuỳ chọn) Lưu về Google Drive để không mất khi Colab ngắt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/DATN_models"
!cp runs/detect/{NAME}/weights/best.pt "/content/drive/MyDrive/DATN_models/{NAME}_best.pt"
print("Đã lưu vào Google Drive/DATN_models")